In [ ]:
# 📦 Dependency Installation (for Colab or local environments)
try:
    import chromadb
    from sentence_transformers import SentenceTransformer
    import pypdf
    import ollama
    print("✅ Standard dependencies found.")
except ImportError:
    print("🚀 Installing missing dependencies...")
    !pip install -q chromadb sentence-transformers pypdf ollama ipywidgets tqdm
    print("✅ Installation complete. Please restart the kernel if imports still fail.")

# 📥 Ingestion Pipeline
### *Data Processing and Vector Database Construction*

This notebook handles the conversion of PDF design guidelines into a searchable vector database using **ChromaDB** and **Sentence Transformers**.

## 1. Setup & Configuration
Initializing paths and loading the embedding model.

In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
import pypdf
from tqdm.auto import tqdm

# Config
DATA_DIR = "../data"
CHROMA_PATH = "../chroma_db"
COLLECTION_NAME = "design_guides"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

print("🔹 Loading embedding model...")
model = SentenceTransformer(EMBEDDING_MODEL)

print("🔹 Connecting to ChromaDB...")
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(name=COLLECTION_NAME)

print("✅ Configuration Loaded.")

## 2. Document Processing
Extracting text from PDF files in the `data/` directory.

In [ ]:
def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = pypdf.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

documents = []
pdf_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.pdf')]

for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
    path = os.path.join(DATA_DIR, pdf_file)
    content = extract_text_from_pdf(path)
    documents.append({"name": pdf_file, "content": content})

print(f"✅ Extracted text from {len(documents)} documents.")

## 3. Chunking & Indexing
Splitting text into manageable chunks and inserting into ChromaDB.

In [ ]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

def get_chunks(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i + size])
    return chunks

print("🚀 Starting Indexing...")
for doc in documents:
    chunks = get_chunks(doc["content"])
    
    ids = [f"{doc['name']}_{i}" for i in range(len(chunks))]
    metadatas = [{"source": doc["name"]} for _ in range(len(chunks))]
    embeddings = model.encode(chunks).tolist()
    
    collection.add(
        ids=ids,
        documents=chunks,
        metadatas=metadatas,
        embeddings=embeddings
    )
    print(f"🔹 Indexed {len(chunks)} chunks from {doc['name']}")

print(f"✅ All documents indexed. Total count: {collection.count()}")